# House Price Prediction — Model Training

This notebook documents the model-training workflow for the House
Price Prediction project: loading the California Housing dataset,
splitting it, fitting preprocessing safely (on the training data
only), training five regression models, and comparing their test-set
performance.

**This notebook does not save any `.pkl` files, does not perform
hyperparameter tuning, and does not build the Streamlit dashboard**
— it's a clean, documented run of the core training workflow.


## 1. Imports and Load the Data

We load the dataset with `fetch_california_housing(as_frame=True)`,
which returns a single DataFrame with the 8 feature columns plus the
target, `MedHouseVal`.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

housing_bunch = fetch_california_housing(as_frame=True)
housing_df = housing_bunch.frame

print(f"Dataset loaded: {housing_df.shape[0]} rows, {housing_df.shape[1]} columns.")
housing_df.head()


Dataset loaded: 20640 rows, 9 columns.


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 2. Separate X and y


In [ ]:
X = housing_df.drop(columns=["MedHouseVal"])
y = housing_df["MedHouseVal"]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (20640, 8)
y shape: (20640,)


## 3. Train/Test Split

An 80/20 split with `random_state=42` for reproducibility.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)


X_train shape: (16512, 8)
X_test shape:  (4128, 8)


## 4. Fit Preprocessing ONLY on X_train

To avoid data leakage, the preprocessing pipeline (median imputation
+ standard scaling) is fit exclusively on `X_train`. Its learned
statistics (median values, mean, standard deviation) must never come
from `X_test`.


In [ ]:
preprocessing_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessing_pipeline.fit(X_train)
print("Preprocessing pipeline fitted on X_train only.")
print("Steps:", list(preprocessing_pipeline.named_steps.keys()))


Preprocessing pipeline fitted on X_train only.
Steps: ['imputer', 'scaler']


## 5. Transform X_train and X_test

The fitted pipeline is now used to `.transform()` both sets — it is
never re-fit on `X_test`.


In [ ]:
X_train_processed = preprocessing_pipeline.transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape: ", X_test_processed.shape)


X_train_processed shape: (16512, 8)
X_test_processed shape:  (4128, 8)


## 6. Train Five Regression Models

Linear Regression, Ridge, Random Forest, Gradient Boosting, and
XGBoost, each with simple, untuned default-style settings (no
hyperparameter search).


In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1,
    ),
}

trained_models = {}
for model_name, model in models.items():
    model.fit(X_train_processed, y_train)
    trained_models[model_name] = model
    print(f"Trained: {model_name}")


Trained: LinearRegression
Trained: Ridge


Trained: RandomForest


Trained: GradientBoosting
Trained: XGBoost


## 7. Generate Predictions for X_test


In [ ]:
predictions = {
    model_name: model.predict(X_test_processed)
    for model_name, model in trained_models.items()
}

for model_name, y_pred in predictions.items():
    print(f"{model_name}: generated {len(y_pred)} predictions.")


LinearRegression: generated 4128 predictions.
Ridge: generated 4128 predictions.
RandomForest: generated 4128 predictions.
GradientBoosting: generated 4128 predictions.
XGBoost: generated 4128 predictions.


## 8. Calculate Metrics: MAE, MSE, RMSE, R², MAPE


In [ ]:
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "MAPE": mape}

metrics_by_model = {
    model_name: calculate_metrics(y_test, y_pred)
    for model_name, y_pred in predictions.items()
}

for model_name, metrics in metrics_by_model.items():
    print(model_name, metrics)


LinearRegression {'MAE': 0.5332001304956565, 'MSE': 0.5558915986952442, 'RMSE': np.float64(0.7455813830127763), 'R2': 0.575787706032451, 'MAPE': 31.952187413615203}
Ridge {'MAE': 0.5331931195789734, 'MSE': 0.5558548589435969, 'RMSE': np.float64(0.7455567442814778), 'R2': 0.5758157428913686, 'MAPE': 31.951174471561206}
RandomForest {'MAE': 0.3274252027374032, 'MSE': 0.255169737347244, 'RMSE': np.float64(0.5051432839771741), 'R2': 0.8052747336256919, 'MAPE': 18.90904511069277}
GradientBoosting {'MAE': 0.37165044848436773, 'MSE': 0.29399901242474274, 'RMSE': np.float64(0.5422167577867202), 'R2': 0.7756433164710084, 'MAPE': 21.526554317967967}
XGBoost {'MAE': 0.3679089451869144, 'MSE': 0.29003434141296003, 'RMSE': np.float64(0.538548364971021), 'R2': 0.7786688383329732, 'MAPE': 21.31979699137801}


## 9. Model Comparison DataFrame


In [ ]:
comparison_df = (
    pd.DataFrame(metrics_by_model)
    .T
    .rename_axis("Model")
    .reset_index()
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)
comparison_df


,Model,MAE,MSE,RMSE,R2,MAPE
0,RandomForest,0.327425,0.255170,0.505143,0.805275,18.909045
1,XGBoost,0.367909,0.290034,0.538548,0.778669,21.319797
2,GradientBoosting,0.371650,0.293999,0.542217,0.775643,21.526554
3,Ridge,0.533193,0.555855,0.745557,0.575816,31.951174
4,LinearRegression,0.533200,0.555892,0.745581,0.575788,31.952187


## 10. Identify the Best Model (Highest R²)


In [ ]:
best_row = comparison_df.loc[comparison_df["R2"].idxmax()]
best_model_name = best_row["Model"]
best_r2_score = best_row["R2"]

print(f"Best model: {best_model_name}")
print(f"Best R2 score: {best_r2_score:.4f}")


Best model: RandomForest
Best R2 score: 0.8053


## 11. Conclusion

Based on the actual comparison table above:

- **Random Forest was the best-performing model**, with R² ≈ **0.805**
  — meaning it explains about 80.5% of the variance in `MedHouseVal`
  on the held-out test set. It also had the lowest MAE (≈0.327,
  i.e. about $32,700) and lowest RMSE (≈0.505) of all five models.
- **XGBoost (R² ≈ 0.779) and Gradient Boosting (R² ≈ 0.776)** came
  next, close behind Random Forest and close to each other — all
  three tree-based ensemble models clearly outperformed the two
  linear models.
- **Ridge (R² ≈ 0.576) and Linear Regression (R² ≈ 0.576)** were
  essentially tied with each other, and clearly the weakest of the
  five. `Ridge`'s L2 regularization (`alpha=1.0`) made almost no
  difference here — expected, since regularization mainly helps with
  overfitting or highly correlated features, neither of which is a
  major issue for this small, already-scaled 8-feature dataset.
- **The gap between linear and tree-based models (R² ≈ 0.58 vs.
  ≈ 0.78–0.81) suggests the true relationship between these features
  and house price is meaningfully non-linear** — consistent with
  what earlier EDA found: location (`Latitude`/`Longitude`) affects
  price in clusters tied to specific metro areas (e.g. the Bay Area,
  Los Angeles) rather than as a smooth linear gradient, a pattern
  tree-based models can capture but a linear model cannot.
- All five models were trained with simple, untuned settings — no
  hyperparameter search was performed here, so these numbers
  represent a reasonable *baseline* for each model type, not their
  best possible performance.
